In [216]:
from openpyxl import load_workbook
from pandas import ExcelWriter
import numpy as np
import pandas as pd
import requests

In [218]:
p = 'https://www.ons.gov.uk/file?uri=%2fbusinessindustryandtrade%2fretailindustry%2fdatasets%2fretailsalesindexinternetsales%2fcurrent/internetsalesreferencetables.xlsx'
g = 'https://www.ons.gov.uk/file?uri=%2fbusinessindustryandtrade%2fretailindustry%2fdatasets%2fpoundsdatatotalretailsales%2fcurrent/poundsdataaccessible.xlsx'

with open(f"ons_xlsx_total.xlsx", "wb") as f:
    f.write(requests.get(g).content)

with open(f"ons_xlsx_online.xlsx", "wb") as f:
    f.write(requests.get(p).content)

In [219]:
def format_ons_xlsx_online():
    wb = load_workbook('ons_xlsx_online.xlsx')
    ws = wb['IntValSA'].values
    x = []
    df = pd.DataFrame(ws).iloc[3:].reset_index(drop=True)
    header = df.iloc[0]
    df = df[1:]
    df.columns = header
    df = df.dropna(axis='columns').iloc[2:].reset_index(drop=True)
    df['Time Period'] = pd.to_datetime(df['Time Period'], format='%d%b%Y:%H:%M:%S.%f')
    df = df.set_index('Time Period')
    df = df * (52/12)
    df.columns = (df.columns.str.split('[')).str[0]
    df.columns = map(str.lower, df.columns)
    df.columns = df.columns.str.strip()
    df.to_excel('ons_xlsx_online-edited.xlsx')
    return pd.DataFrame(df)

format_ons_xlsx_online()

,all retailing excluding automotive fuel,predominantly food stores,total of predominantly non-food stores,non-specialised stores,"textile, clothing and footwear stores",household goods stores,other stores,non-store retailing
Time Period,,,,,,,,
2008-01-01,965.9,170.3,396.5,71.066667,102.7,94.466667,128.266667,399.1
2008-02-01,1022.233333,169.866667,426.833333,77.133333,113.533333,101.4,134.766667,425.966667
2008-03-01,1046.5,170.733333,432.033333,78.0,114.833333,104.866667,134.333333,443.3
2008-04-01,1098.5,178.1,452.833333,80.6,118.3,110.5,143.433333,467.133333
2008-05-01,1112.8,182.0,458.033333,83.633333,120.9,112.233333,141.266667,473.2
...,...,...,...,...,...,...,...,...
2021-09-01,9695.4,1556.966667,3520.833333,740.566667,1062.966667,729.733333,987.566667,4617.6
2021-10-01,9606.566667,1533.566667,3537.733333,731.466667,1083.333333,721.5,1001.866667,4535.266667
2021-11-01,9535.066667,1476.366667,3513.9,722.366667,1061.666667,720.633333,1009.666667,4544.8


In [221]:

def format_ons_xlsx_total():
    wb = load_workbook('ons_xlsx_total.xlsx')
    ws = wb['ValSAT'].values
    x = []
    df = pd.DataFrame(ws).iloc[4:].reset_index(drop=True)
    header = df.iloc[0]
    df = df[171:]
    df.columns = header
    df = df.drop(['Automotive fuel','All retailing including automotive fuel','Month as a % of Total','Month as a % of Total excluding automotive fuel', 'Total Annual Sales for All Retailing including automotive fuel','Total Annual Sales for All Retailing excluding automotive fuel'], axis=1)
    df = df.rename(columns={df.columns[0]: 'Time Period'}).reset_index(drop=True)
    df = df.rename(columns={df.columns[3]: 'Total of predominantly non-food stores'})
    df['Time Period'] = pd.to_datetime(df['Time Period'])
    df = df.set_index('Time Period')
    df = df.dropna()
    df = df / 1000
    df.columns = map(str.lower, df.columns)
    df.columns = df.columns.str.strip()
    df.to_excel('ons_xlsx_total-edited.xlsx')
    return pd.DataFrame(df)

format_ons_xlsx_total()

,all retailing excluding automotive fuel,predominantly food stores,total of predominantly non-food stores,non-specialised stores,"textile, clothing and footwear stores",household goods stores,other stores,non-store retailing
Time Period,,,,,,,,
2008-01-01,26567.521,11928.874,13375.078,2316.029,3586.613,3345.382,4127.054,1263.569
2008-02-01,21598.888,9585.812,10988.53,1864.443,3019.065,2678.864,3426.158,1024.546
2008-03-01,26722.774,11964.731,13475.545,2255.299,3621.299,3349.491,4249.456,1282.498
2008-04-01,21264.725,9583.67,10666.689,1817.788,2822.724,2652.357,3373.82,1014.366
2008-05-01,22105.547,9874.945,11201.178,1827.872,3178.13,2718.318,3476.858,1029.424
...,...,...,...,...,...,...,...,...
2021-09-01,40382.277,16935.059,17110.136,3261.015,4416.558,3485.059,5947.504,6337.082
2021-10-01,33029.81,13698.829,14274.773,2614.155,3762.978,2867.413,5030.227,5056.208
2021-11-01,33399.809,13747.921,14464.685,2573.375,3867.742,2928.146,5095.422,5187.203


In [238]:
def format_ons_xlsx_offline():
    return format_ons_xlsx_total() - format_ons_xlsx_online()

def oao_process(format1):
    format1['yoy % all retailing excluding automotive fuel'] = format1['all retailing excluding automotive fuel'].pct_change(periods=12) * 100
    return format1[['yoy % all retailing excluding automotive fuel','predominantly food stores','total of predominantly non-food stores','non-store retailing']].dropna()

def ovo_process(format1):
    format1['yoy % all retailing excluding automotive fuel'] = format1['all retailing excluding automotive fuel'].pct_change(periods=12) * 100
    return format1[['yoy % all retailing excluding automotive fuel','all retailing excluding automotive fuel',]].dropna()

with pd.ExcelWriter('output.xlsx') as writer:  
    format_ons_xlsx_total().to_excel(writer, sheet_name='TotalValSAT')
    format_ons_xlsx_online().to_excel(writer, sheet_name='IntValSAT')
    format_ons_xlsx_offline().to_excel(writer, sheet_name='OfflineValSAT')
    oao_process(format_ons_xlsx_online()).to_excel(writer, sheet_name='online_by_category-flourish')
    oao_process(format_ons_xlsx_offline()).to_excel(writer, sheet_name='offline_by_category-flourish')
    ovo_process(format_ons_xlsx_online()).to_excel(writer, sheet_name='online_yoy_total-flourish')
    ovo_process(format_ons_xlsx_offline()).to_excel(writer, sheet_name='offline_yoy_total-flourish')
    

# What format does Flourish require?

## The charts
**Online and offline**
 - Total (YoY)	Food	Non-food	Non-store
 - t.csv, r.csv

**Online vs offline**
 - Online YoY%	Offline YoY%	Online Sales	Offline Sales
 - q.csv

**YoY % by Category**
 - Total (excl. Fuel)	Food	Non-food	Non-store
 - aggregate_df_total.csv